In [2]:
import json
import pandas as pd
import numpy as np
from glob import glob
import os   
from collections import defaultdict
from pprint import pprint
import torch
from huggingface_hub import login
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split


/Users/shan95/anaconda3/envs/lus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def get_device():
    """Detect best available device on Mac"""
    if torch.cuda.is_available():
        device = "cuda"
        print("✅ Using CUDA GPU")
    elif torch.backends.mps.is_available():
        device = "mps"
        print("✅ Using Apple Silicon MPS")
    else:
        device = "cpu"
        print("⚠️  Using CPU (this will be slow)")
    
    return device




In [4]:
TAXONOMY = {
    'A': {
        'adaptive': [
            '(13) Feel loved, belong',
            '(9) Justifiable anger/ assertive anger, justifiable outrage',
            '(11) Proud',
            '(7) Vigor / energetic',
            '(5) Content, happy, joy, hopeful',
            '(1) Calm/ laid back',
            '(3) Sad, emotional pain, grieving'
            ],
        'maladaptive': [
            '(12) Ashamed, guilty',
            '(10) Angry (aggression), disgust, contempt',
            '(8) Apathic, don’t care, blunted',
            '(4) Depressed, despair, hopeless',
            '(6) Mania',
            '(14) Feel lonely',
            '(2) Anxious/ fearful/ tense'
            ]
          },
    'B-S': {
        'adaptive': [
            '(1) Self care and improvement'
            ],
        'maladaptive': [
            '(2) Self harm, neglect and avoidance'
            ]
    },
    'B-O': {
        'adaptive': 
            [
            '(3) Autonomous or adaptive control behavior',
            '(1) Relating behavior'
            ],
        'maladaptive': [
            '(2) Fight or flight behavior',
            '(4) Over controlled or controlling behavior'
            ]
    },
    'C-S': 
        {
        'adaptive': 
            [
            '(1) Self-acceptance and compassion'
            ],
          'maladaptive': 
              [
              '(2) Self criticism'
              ]
        },
    'C-O': 
        {
            'adaptive': 
            [
                '(1) Perception of the other as related',
                '(3) Perception of the other as facilitating autonomy needs'
            ],
              'maladaptive': 
              [
                  '(4) Perception of the other as blocking autonomy needs',
                  '(2) Perception of the other as detached or over attached'
                  ]
          },
    'D': 
    {
        'adaptive': 
            [
            '(5) Competence, self esteem, self-care',
            '(1) Relatedness',
            '(3) Autonomy and adaptive control'
            ],
      'maladaptive': 
          [
              '(6) Expectation that competence needs will not be met',
              '(4) Expectation that autonomy needs will not be met',
              '(2) Expectation that relatedness needs will not be met'
          ]
    }
}


def get_taxonomy_string():
    """Format taxonomy for prompt"""
    lines = []
    for dim, categories in TAXONOMY.items():
        lines.append(f"\n**{dim}**")
        lines.append("Adaptive:")
        for cat in categories['adaptive']:
            lines.append(f"  - {cat}")
        lines.append("Maladaptive:")
        for cat in categories['maladaptive']:
            lines.append(f"  - {cat}")
    return "\n".join(lines)

In [5]:
class CLPsychDataLoader:
    """Load CLPsych data with proper ordering"""
    
    def __init__(self, input_dir):
        self.input_dir = input_dir
        self.df = None
    
    def load_clpsych_data(self, filepath):
        with open(filepath, 'r') as f:
            data = json.load(f)
            id, post = data['timeline_id'], data['posts']
        return id, post

    def load(self):
        """Load and parse JSON data into sorted DataFrame"""

        train_posts = []
        for file in glob(os.path.join(self.input_dir, '*.json')):
            # print(f"Loading {file}...")
            id, posts = self.load_clpsych_data(file)
            # print(f"Loaded {id} with {len(posts)} posts.")
            for post in posts:
                # print(post)
                try:
                    assert 'post_id' in post
                    assert 'post' in post
                    assert 'evidence' in post
                except AssertionError:
                    print(f"Timeline {id}, Post {post['post_id']} is missing required fields.")
                    continue
                train_posts.append({
                    'timeline_id': id,
                    'post_id': post['post_id'],
                    'post_index': post['post_index'],
                    'text': post['post'],
                    'well_being': post.get('Well-being', 0),
                    'is_switch': post.get('Switch', 0),
                    'is_escalation': post.get('Escalation', 0),
                    'evidence': post['evidence']
                })
        
        # Create DataFrame and sort by timeline_id and post_index
        self.df = pd.DataFrame(train_posts)
        self.df = self.df.sort_values(['timeline_id', 'post_index'])
        
        print(f"Loaded {len(self.df)} posts from {self.df['timeline_id'].nunique()} timelines")
        
        return self.df
    
    def verify_order(self):
        """Verify posts are in correct order within each timeline"""
        print("\n=== Verifying Post Order ===")
        issues = []
        
        for timeline_id in self.df['timeline_id'].unique():
            timeline_posts = self.df[self.df['timeline_id'] == timeline_id]
            indices = timeline_posts['post_index'].tolist()
            
            # Check if indices are in ascending order
            if indices != sorted(indices):
                issues.append(f"Timeline {timeline_id}: {indices}")
        
        if issues:
            print("❌ Order issues found:")
            for issue in issues:
                print(f"  {issue}")
            return False
        else:
            print("✅ All posts are in correct order")
            return True
    
    def get_stats(self):
        """Print dataset statistics"""
        print("\n=== Dataset Statistics ===")
        print(f"Total timelines: {self.df['timeline_id'].nunique()}")
        print(f"Total posts: {len(self.df)}")
        print(f"Avg posts per timeline: {len(self.df) / self.df['timeline_id'].nunique():.2f}")
        
        # print(f"\nSwitch events: {self.df['is_switch'].sum()} ({self.df['is_switch'].mean()*100:.1f}%)")
        # print(f"Escalation events: {self.df['is_escalation'].sum()} ({self.df['is_escalation'].mean()*100:.1f}%)")
        
        # ABCD presence
        print("\n=== ABCD Element Presence ===")
        adaptive_counts = defaultdict(int)
        maladaptive_counts = defaultdict(int)
        
        for _, row in self.df.iterrows():
            evidence = row['evidence']
            
            # Adaptive
            if 'adaptive-state' in evidence:
                for dim in ['A', 'B-S', 'B-O', 'C-S', 'C-O', 'D']:
                    if dim in evidence['adaptive-state'] and evidence['adaptive-state'][dim].get('Category'):
                        adaptive_counts[dim] += 1
            
            # Maladaptive
            if 'maladaptive-state' in evidence:
                for dim in ['A', 'B-S', 'B-O', 'C-S', 'C-O', 'D']:
                    if dim in evidence['maladaptive-state'] and evidence['maladaptive-state'][dim].get('Category'):
                        maladaptive_counts[dim] += 1
        
        print("\nAdaptive:")
        for dim in ['A', 'B-S', 'B-O', 'C-S', 'C-O', 'D']:
            print(f"  {dim}: {adaptive_counts[dim]}")
        
        print("\nMaladaptive:")
        for dim in ['A', 'B-S', 'B-O', 'C-S', 'C-O', 'D']:
            print(f"  {dim}: {maladaptive_counts[dim]}")

# Load data
# loader = CLPsychDataLoader('train_tasks12/')
# df = loader.load()
# loader.verify_order()
# loader.get_stats()

In [6]:
def format_evidence_as_json(evidence):
    """Convert evidence dict to clean JSON string"""
    output = {
        'adaptive-state': {},
        'maladaptive-state': {}
    }
    
    # Process adaptive state
    if 'adaptive-state' in evidence:
        for dim, data in evidence['adaptive-state'].items():
            if dim != 'Presence' and isinstance(data, dict):
                if 'Category' in data and data['Category']:
                    output['adaptive-state'][dim] = {
                        'Category': data['Category'],
                        'highlighted_evidence': data.get('highlighted_evidence', '')
                    }
    
    # Process maladaptive state
    if 'maladaptive-state' in evidence:
        for dim, data in evidence['maladaptive-state'].items():
            if dim != 'Presence' and isinstance(data, dict):
                if 'Category' in data and data['Category']:
                    output['maladaptive-state'][dim] = {
                        'Category': data['Category'],
                        'highlighted_evidence': data.get('highlighted_evidence', '')
                    }
    
    return json.dumps(output, indent=2)

def create_instruction_dataset(df):
    """
    Convert DataFrame to instruction-tuning format
    Maintains order from sorted DataFrame
    """
    
    instruction = """You are an expert in mental health text analysis using the MIND framework. Analyze the social media post and identify ABCD (Affect, Behavior, Cognition, Desire) elements.

For each post, identify:
- Adaptive self-state elements (supporting well-being)
- Maladaptive self-state elements (negative patterns)

ABCD Dimensions:
- A: Affect (emotional tone)
- B-S: Behavior toward Self
- B-O: Behavior toward Others
- C-S: Cognition of Self (beliefs about self)
- C-O: Cognition of Others (perceptions of others)
- D: Desire (motivations, needs, expectations)

Categories:
""" + get_taxonomy_string() + """

Output JSON format:
{
  "adaptive-state": {
    "dimension": {
      "Category": "(number) category name",
      "highlighted_evidence": "exact quote from post"
    }
  },
  "maladaptive-state": {
    "dimension": {
      "Category": "(number) category name",
      "highlighted_evidence": "exact quote from post"
    }
  }
}

Rules:
1. Each dimension can appear in adaptive, maladaptive, both, or neither
2. Evidence must be exact quote from the post (3-15 words)
3. Only include dimensions you detect in the post
4. Multiple dimensions can be present simultaneously"""

    dataset = []
    
    # Iterate through sorted DataFrame
    for idx, row in df.iterrows():
        # Skip posts without evidence
        if not row.get('evidence'):
            continue
        
        # Check if there's actual content
        has_content = False
        evidence = row['evidence']
        
        for state in ['adaptive-state', 'maladaptive-state']:
            if state in evidence:
                for dim, data in evidence[state].items():
                    if dim != 'Presence' and isinstance(data, dict):
                        if data.get('Category'):
                            has_content = True
                            break
        
        if not has_content:
            continue
        
        dataset.append({
            'timeline_id': row['timeline_id'],
            'post_index': row['post_index'],
            'post_id': row['post_id'],
            'instruction': instruction,
            'input': f"Post: {row['text']}",
            'output': format_evidence_as_json(row['evidence'])
        })
    
    # Convert back to DataFrame to maintain order
    dataset_df = pd.DataFrame(dataset)
    dataset_df = dataset_df.sort_values(['timeline_id', 'post_index']).reset_index(drop=True)
    
    print(f"\nCreated {len(dataset_df)} instruction examples")
    print(f"From {dataset_df['timeline_id'].nunique()} timelines")
    
    return dataset_df

# Create instruction dataset
# instruction_df = create_instruction_dataset(df)

# Verify first few examples
# print("\n=== First 3 Examples ===")
# for idx in range(min(3, len(instruction_df))):
#     row = instruction_df.iloc[idx]
#     print(f"\nExample {idx+1}:")
#     print(f"Timeline: {row['timeline_id']}, Post: {row['post_index']}")
#     print(f"Input: {row['input'][:100]}...")
#     print(f"Output: {row['output'][:200]}...")

In [7]:
from sklearn.model_selection import train_test_split

def timeline_aware_split(instruction_df, test_size=0.15, random_state=42):
    """
    Split by timeline while preserving post order
    """
    # Get unique timeline IDs
    timeline_ids = instruction_df['timeline_id'].unique().tolist()
    
    # Split timeline IDs
    train_timeline_ids, val_timeline_ids = train_test_split(
        timeline_ids,
        test_size=test_size,
        random_state=random_state
    )
    
    # Filter by timeline
    train_df = instruction_df[instruction_df['timeline_id'].isin(train_timeline_ids)].copy()
    val_df = instruction_df[instruction_df['timeline_id'].isin(val_timeline_ids)].copy()
    
    # Ensure order is maintained
    train_df = train_df.sort_values(['timeline_id', 'post_index']).reset_index(drop=True)
    val_df = val_df.sort_values(['timeline_id', 'post_index']).reset_index(drop=True)
    
    print(f"\n=== Train/Val Split ===")
    print(f"Train: {len(train_df)} posts from {len(train_timeline_ids)} timelines")
    print(f"Val: {len(val_df)} posts from {len(val_timeline_ids)} timelines")
    
    # Verify order in both sets
    print("\nVerifying order in train set...")
    for timeline_id in train_timeline_ids[:2]:
        timeline_posts = train_df[train_df['timeline_id'] == timeline_id]
        indices = timeline_posts['post_index'].tolist()
        print(f"  Timeline {timeline_id}: {indices}")
        assert indices == sorted(indices), f"Order broken in train for {timeline_id}!"
    
    print("\nVerifying order in val set...")
    for timeline_id in val_timeline_ids[:2]:
        timeline_posts = val_df[val_df['timeline_id'] == timeline_id]
        indices = timeline_posts['post_index'].tolist()
        print(f"  Timeline {timeline_id}: {indices}")
        assert indices == sorted(indices), f"Order broken in val for {timeline_id}!"
    
    print("✅ Order verified in both train and val sets")
    
    return train_df, val_df

# Split data
# train_df, val_df = timeline_aware_split(instruction_df, test_size=0.15)

In [8]:
def df_to_training_format(df):
    """
    Convert DataFrame to list of dicts for training
    Order is already preserved in DataFrame
    """
    training_data = []
    
    for idx, row in df.iterrows():
        training_data.append({
            'instruction': row['instruction'],
            'input': row['input'],
            'output': row['output']
        })
    
    return training_data

# Convert to training format
# train_data = df_to_training_format(train_df)
# val_data = df_to_training_format(val_df)

# print(f"\nTrain data: {len(train_data)} examples")
# print(f"Val data: {len(val_data)} examples")

# # Save for inspection
# print("\n=== Saving sample data ===")
# with open('train_sample.json', 'w') as f:
#     json.dump(train_data[:5], f, indent=2)
print("Saved first 5 training examples to train_sample.json")

Saved first 5 training examples to train_sample.json


In [9]:
from torch.utils.data import Dataset
import torch

class ABCDInstructionDataset(Dataset):
    """
    Dataset for ABCD instruction tuning
    Preserves order from input list
    """
    
    def __init__(self, data, tokenizer, max_length=2048):
        """
        Args:
            data: List of dicts with 'instruction', 'input', 'output'
                  (already in correct order)
            tokenizer: Hugging Face tokenizer
            max_length: Maximum sequence length
        """
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        """Get item at index (maintains order)"""
        item = self.data[idx]
        
        # Format as Llama-3 chat format
        messages = [
            {"role": "system", "content": item['instruction']},
            {"role": "user", "content": item['input']},
            {"role": "assistant", "content": item['output']}
        ]
        
        # Apply chat template
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        
        # Tokenize
        encodings = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze(),
            'labels': encodings['input_ids'].squeeze()
        }

# Note: We'll create the actual dataset after loading the model
# to avoid loading tokenizer twice

In [ ]:
device = get_device()
hf_token = "hf"
login(token=hf_token)

# ========== STEP 1: Load Data ==========
print("=" * 60)
print("STEP 1: Loading Data")
print("=" * 60)

loader = CLPsychDataLoader('train_tasks12/')
df = loader.load()
loader.verify_order()
loader.get_stats()

# ========== STEP 2: Create Instruction Dataset ==========
print("\n" + "=" * 60)
print("STEP 2: Creating Instruction Dataset")
print("=" * 60)

instruction_df = create_instruction_dataset(df)

# ========== STEP 3: Train/Val Split ==========
print("\n" + "=" * 60)
print("STEP 3: Train/Val Split")
print("=" * 60)

train_df, val_df = timeline_aware_split(instruction_df, test_size=0.15)

# Convert to list format
train_data = df_to_training_format(train_df)
val_data = df_to_training_format(val_df)

# ========== STEP 4: Load Model ==========
print("\n" + "=" * 60)
print("STEP 4: Loading Llama-3-8B Model")
print("=" * 60)

model_name = "meta-llama/Llama-3.1-8B-Instruct"

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model WITHOUT BitsAndBytesConfig
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = model.to(device)
model.config.use_cache = False
model.config.pretraining_tp = 1

# ========== STEP 5: Setup LoRA ==========
print("\n" + "=" * 60)
print("STEP 5: Configuring LoRA")
print("=" * 60)

model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=8,  # Rank
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ========== STEP 6: Create Datasets ==========
print("\n" + "=" * 60)
print("STEP 6: Creating PyTorch Datasets")
print("=" * 60)

train_dataset = ABCDInstructionDataset(train_data, tokenizer, max_length=1024)
val_dataset = ABCDInstructionDataset(val_data, tokenizer, max_length=1024)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset: {len(val_dataset)} examples")

# ========== STEP 7: Training Configuration ==========
print("\n" + "=" * 60)
print("STEP 7: Configuring Training")
print("=" * 60)

# Add this right before training
print("\n" + "="*60)
print("FINAL DEVICE CHECK")
print("="*60)
print(f"Model device: {next(model.parameters()).device}")
print(f"Expected: mps")

# Quick speed test
import time
device = next(model.parameters()).device

x = torch.randn(1000, 1000).to(device)
start = time.time()
for _ in range(100):
    y = x @ x
if device.type == "mps":
    torch.mps.synchronize()
elapsed = time.time() - start
print(f"GPU matrix multiply (100 iterations): {elapsed:.2f}s")
print("If > 5s, something is wrong")
print("="*60 + "\n")

training_args = TrainingArguments(
    output_dir="./llama3-abcd-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    group_by_length=True,
    report_to="none",
    save_total_limit=2,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

# ========== STEP 8: Train ==========
print("\n" + "=" * 60)
print("STEP 8: Starting Training")
print("=" * 60)

trainer.train()

# ========== STEP 9: Save Model ==========
print("\n" + "=" * 60)
print("STEP 9: Saving Model")
print("=" * 60)

trainer.save_model("./llama3-abcd-lora-final")
tokenizer.save_pretrained("./llama3-abcd-lora-final")

print("\n✅ Training complete! Model saved to ./llama3-abcd-lora-final")

✅ Using Apple Silicon MPS
STEP 1: Loading Data
Loaded 373 posts from 30 timelines

=== Verifying Post Order ===
✅ All posts are in correct order

=== Dataset Statistics ===
Total timelines: 30
Total posts: 373
Avg posts per timeline: 12.43

=== ABCD Element Presence ===

Adaptive:
  A: 43
  B-S: 84
  B-O: 125
  C-S: 52
  C-O: 50
  D: 118

Maladaptive:
  A: 161
  B-S: 59
  B-O: 36
  C-S: 135
  C-O: 114
  D: 110

STEP 2: Creating Instruction Dataset

Created 236 instruction examples
From 30 timelines

STEP 3: Train/Val Split

=== Train/Val Split ===
Train: 195 posts from 25 timelines
Val: 41 posts from 5 timelines

Verifying order in train set...
  Timeline 6c9677b482: [2, 3, 4, 5, 7, 9, 10]
  Timeline d237ea4269: [1, 2, 3, 4, 5, 6, 10, 11, 12]

Verifying order in val set...
  Timeline d0fb4b962e: [1, 2, 3, 4, 5, 6, 7, 9, 10]
  Timeline 8e0a58cfbd: [4, 5, 6, 8, 9, 10, 12, 16, 18, 19]
✅ Order verified in both train and val sets

STEP 4: Loading Llama-3-8B Model


Fetching 4 files:   0%|          | 0/4 [00:35<?, ?it/s]
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 